In [49]:
# -*- coding: utf-8 -*-
"""
BTC 2025 顶部“模糊正确”预测（时间 / 价格 / Ahr999 三模型 + 交叉验证）

使用方法：
1) 直接运行即可得到一套默认结果（无需时间序列 CSV）。
2) 若你有 Coinglass 导出的逐日数据（包含列: date, price, ahr999, ma200），
   把 SERIES_CSV 指向该文件路径，脚本会自动用 10/20 日窗口做平滑并提高稳健性。
"""

from __future__ import annotations
from dataclasses import dataclass
from datetime import date, timedelta, datetime
from typing import List, Optional, Tuple, Dict
import math
import numpy as np
import csv
from statistics import mean, median

# ========== 0) 原始锚点（你给的“很重要”的日期与结论） ==========
HALVINGS: List[date] = [
    date(2012, 11, 28),
    date(2016, 7, 9),
    date(2020, 5, 11),
    date(2024, 4, 20),  # 最近一次
]
TOPS: List[date] = [               # 各减半后的当轮大顶
    date(2013, 11, 29),
    date(2017, 12, 17),
    date(2021, 11, 10),
]
BOTTOMS: List[date] = [            # 上一轮熊底（用于“底→减半”的稳定性检验）
    date(2015, 1, 14),
    date(2018, 12, 15),
    date(2022, 11, 21),
]

# 历史大顶的 Ahr999（你给的截图精确值）
AHR999_PEAKS = {
    date(2013, 11, 27): 55,
    date(2017, 12, 19): 15,
    date(2021, 11, 8):  3.3,
}


@dataclass
class MultiModelResult:
    base_days: int
    base_date: date
    regress_days: int
    regress_date: date
    peak2peak_days: int
    peak2peak_date: date
    window: tuple[date, date]

def predict_peak_multi() -> MultiModelResult:
    # --- 1) 基线：减半→顶 中位数 ---
    halving_to_top = [(TOPS[i] - HALVINGS[i]).days for i in range(len(TOPS))]
    base_days = int(round(median(halving_to_top)))
    base_date = HALVINGS[-1] + timedelta(days=base_days)

    # --- 2) 修正 A：回归（减半间隔 → 延迟） ---
    halving_intervals = [(HALVINGS[i+1] - HALVINGS[i]).days for i in range(len(HALVINGS)-1)]
    x = np.array(halving_intervals, dtype=float)
    y = np.array(halving_to_top, dtype=float)
    A = np.vstack([x, np.ones(len(x))]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]
    regress_days = int(round(a * halving_intervals[-1] + b))
    regress_date = HALVINGS[-1] + timedelta(days=regress_days)

    # --- 3) 修正 B：顶→顶 趋势 ---
    peak2peak = [(TOPS[i+1] - TOPS[i]).days for i in range(len(TOPS)-1)]
    avg_p2p = int(round(np.mean(peak2peak)))
    peak2peak_date = TOPS[-1] + timedelta(days=avg_p2p)
    # 转换成“相对 2024 减半”的延迟天数
    peak2peak_days = (peak2peak_date - HALVINGS[-1]).days

    # --- 4) 综合窗口 ---
    lo = min(base_date, regress_date, peak2peak_date)
    hi = max(base_date, regress_date, peak2peak_date)

    return MultiModelResult(
        base_days, base_date,
        regress_days, regress_date,
        peak2peak_days, peak2peak_date,
        (lo, hi)
    )

# 测试
if __name__ == "__main__":
    r = predict_peak_multi()
    print("基线（减半→顶中位数）:", r.base_days, "天 →", r.base_date)
    print("修正A（回归预测）  :", r.regress_days, "天 →", r.regress_date)
    print("修正B（顶→顶外推）:", r.peak2peak_days, "天 →", r.peak2peak_date)
    print("综合预测窗口       :", r.window[0], "→", r.window[1])


基线（减半→顶中位数）: 526 天 → 2025-09-28
修正A（回归预测）  : 563 天 → 2025-11-04
修正B（顶→顶外推）: 560 天 → 2025-11-01
综合预测窗口       : 2025-09-28 → 2025-11-04


In [50]:
import numpy as np

# ================== 历史锚点（名义价，仅用于拟合趋势；不在过程中叠加通胀） ==================
BOTTOMS = np.array([150.0, 3200.0, 15500.0])  # 2015, 2018, 2022
TOPS    = np.array([19800.0, 69000.0])        # 2017, 2021

# 派生：底→顶倍数、顶→底回撤比例（比值，不受通胀影响）
mult_hist = np.array([TOPS[0]/BOTTOMS[0], TOPS[1]/BOTTOMS[1]])   # [132, 21]
retr_hist = np.array([BOTTOMS[1]/TOPS[0], BOTTOMS[2]/TOPS[1]])   # [0.16, 0.22]

# ================== 趋势拟合（仍用名义历史，但仅用于“软约束”的方向性） ==================
cycles_m = np.array([2.0, 3.0])                    # 第二、第三轮
# 倍数衰减：log(mult) ~ a_m*cycle + b_m
a_m, b_m = np.polyfit(cycles_m, np.log(mult_hist), 1)
mult_pred_trend = float(np.exp(a_m*4 + b_m))       # 第四轮倍数“趋势值”（~3.3x，软约束）

# 回撤收敛：retr ~ a_r*cycle + b_r
a_r, b_r = np.polyfit(cycles_m, retr_hist, 1)
retr_pred_trend = float(a_r*4 + b_r)               # ~0.27~0.30，软约束

# 顶→顶增长：log(top) ~ a_t*cycle + b_t  （名义价，仅作软约束）
a_t, b_t = np.polyfit(cycles_m, np.log(TOPS), 1)
top_trend_4 = float(np.exp(a_t*4 + b_t))

# 底→底增长：log(bottom) ~ a_b*cycle + b_b（名义价，仅作软约束）
cycles_b = np.array([2.0, 3.0, 4.0])               # 2015, 2018, 2022
a_b, b_b = np.polyfit(cycles_b, np.log(BOTTOMS), 1)
bottom_trend_5 = float(np.exp(a_b*5 + b_b))

# ================== 网格搜索联合拟合（四关系） ==================
B3 = BOTTOMS[-1]                 # 2022 底（作为起点，实值）
curr_price_anchor = 120000.0     # 现价下限（名义观察值，作为硬/软约束用）
hard_floor = True                # True=硬约束丢弃 T4<anchor 的解；False=使用软惩罚

# 约束权重
w_mult   = 0.7     # 倍数贴合衰减趋势（比值）
w_retr   = 0.9     # 回撤贴合收敛趋势（比值）
w_top    = 0.6     # 顶贴合顶→顶趋势（名义）
w_bottom = 0.10    # 底贴合底→底趋势（名义）
w_floor  = 10.0    # 若软约束时的惩罚强度

# 搜索范围：第四轮倍数 & 回撤比例（实值阶段）
mult_grid = np.linspace(4.0, 20.0, 641)    # 4x ~ 20x
retr_grid = np.linspace(0.20, 0.35, 301)   # 0.20 ~ 0.35

best = None
records = []

for M4 in mult_grid:
    T4_real = B3 * M4              # 实值顶（未加通胀）
    # 顶对比“现价锚”用名义值；按你的要求，这里不引入通胀，直接用实值近似比较
    if hard_floor and (T4_real < curr_price_anchor):
        continue
    floor_pen = 0.0
    if (not hard_floor) and (T4_real < curr_price_anchor):
        floor_pen = w_floor * (np.log(curr_price_anchor) - np.log(T4_real))**2

    # 倍数与趋势（比值空间）
    cost_mult = w_mult * (np.log(M4) - np.log(mult_pred_trend))**2
    # 顶与顶趋势（名义趋势 vs 实值顶，近似作为软约束）
    cost_top  = w_top  * (np.log(T4_real) - np.log(top_trend_4))**2

    for R4 in retr_grid:
        B5_real = T4_real * R4     # 实值下一底（未加通胀）

        # 回撤与趋势（比值空间）
        cost_retr = w_retr * (R4 - retr_pred_trend)**2
        # 底与底趋势（名义趋势 vs 实值底，近似作为软约束）
        cost_bot  = w_bottom * (np.log(B5_real) - np.log(bottom_trend_5))**2

        cost = cost_mult + cost_retr + cost_top + cost_bot + floor_pen
        records.append((cost, M4, R4, T4_real, B5_real))
        if (best is None) or (cost < best[0]):
            best = (cost, M4, R4, T4_real, B5_real)

records = np.array(records, dtype=float)
if records.size == 0:
    raise RuntimeError("无可行解（可能硬约束过严或现价锚过高）。请调整 curr_price_anchor 或放宽网格。")

# 形成“模糊正确”带（实值）
min_cost = best[0]
band = records[records[:,0] <= (min_cost * 1.25)]
T4_real_band = band[:,3]
B5_real_band = band[:,4]

# ================== 通胀（最后一步后置校正；年化，按年数幂次） ==================
infl_annual = 0.03       # 年化通胀 1%
years_bottom_to_top = 3.0 # 2022 底 -> 本轮顶，年数（例如 3 年）
years_top_to_next_bottom = 1.2  # 顶 -> 下一底，年数（例如 1.2 年）

def adj_infl(x, annual_rate, years):
    # 同时支持标量和 numpy 数组
    x = np.asarray(x, dtype=float)
    factor = (1.0 + annual_rate) ** years
    return x * factor


# 最优点（名义，通胀后）
T4_nominal_best = adj_infl(best[3], infl_annual, years_bottom_to_top)
B5_nominal_best = adj_infl(best[4], infl_annual, years_bottom_to_top + years_top_to_next_bottom)

# 区间（名义，通胀后）
T4_nominal_band = adj_infl(T4_real_band, infl_annual, years_bottom_to_top)
B5_nominal_band = adj_infl(B5_real_band, infl_annual, years_bottom_to_top + years_top_to_next_bottom)

print("=== 四关系联合拟合（实值） → 最后一步年化通胀校正（名义） ===")
print(f"参数：年化通胀={infl_annual:.2%}；底→顶年数={years_bottom_to_top}；顶→下一底年数={years_top_to_next_bottom}")
print(f"最优(实值)：顶≈${best[3]:,.0f}，下一底≈${best[4]:,.0f}")
print(f"最优(名义)：顶≈${T4_nominal_best:,.0f}，下一底≈${B5_nominal_best:,.0f}")
print(f"模糊区间(名义顶)：${np.min(T4_nominal_band):,.0f}  ~  ${np.max(T4_nominal_band):,.0f}")
print(f"模糊区间(名义底)：${np.min(B5_nominal_band):,.0f}  ~  ${np.max(B5_nominal_band):,.0f}")


=== 四关系联合拟合（实值） → 最后一步年化通胀校正（名义） ===
参数：年化通胀=3.00%；底→顶年数=3.0；顶→下一底年数=1.2
最优(实值)：顶≈$122,062，下一底≈$42,722
最优(名义)：顶≈$133,381，下一底≈$48,369
模糊区间(名义顶)：$131,264  ~  $201,977
模糊区间(名义底)：$27,201  ~  $73,244


In [51]:
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple

# ===== 1) 基本输入：历史锚点（名义价）+ 对应年份 =====
# 注意：这里的价格仍是“名义价”，我们将用通胀率把它们折算到 base_year 的“实际价（Real）”
BOTTOMS_NOMINAL = np.array([150.0, 3200.0, 15500.0])   # 2015, 2018, 2022
BOTTOM_YEARS    = np.array([2015, 2018, 2022])

TOPS_NOMINAL    = np.array([19800.0, 69000.0])         # 2017, 2021
TOP_YEARS       = np.array([2017, 2021])

# 当前“名义价”观测下限（硬/软约束使用），以及其所在年份
CURR_PRICE_ANCHOR_NOMINAL = 120_000.0
CURR_ANCHOR_YEAR = 2025

# ===== 2) 年度通胀率（可以替换为你的真实序列；从 2013 开始）=====
# 例：这里仅示例一组大致路径；你可改成自己的序列（如 CPI 同比）
ANNUAL_INFL: Dict[int, float] = {
    2013: 0.015, 2014: 0.016, 2015: 0.001, 2016: 0.013,
    2017: 0.021, 2018: 0.024, 2019: 0.018, 2020: 0.012,
    2021: 0.047, 2022: 0.080, 2023: 0.041, 2024: 0.029,
    2025: 0.027, 2026: 0.025, 2027: 0.024, 2028: 0.022, 2029: 0.020
}

# ===== 3) 预测年份设定（只用于最后一步把 Real 转回名义）=====
BASE_REAL_YEAR = 2025          # 统一购买力基准（建议用 2025）
PRED_TOP_YEAR = 2025           # 预测顶发生的年份（配合你的时间模型 A）
PRED_NEXT_BOTTOM_YEAR = 2027   # 预测下一熊底的年份

# ===== 4) 工具函数：构建“价格水平指数”，并做通胀折算 =====
def build_price_level_from_rates(annual_rates: Dict[int, float], base_year: int) -> Dict[int, float]:
    """
    返回每年相对 base_year 的价格水平指数 Level[year]，使得 Level[base_year] = 1。
    若 year < base_year，则 Level[year] = 1 / Π_{t=year+1..base} (1+infl[t])
    若 year > base_year，则 Level[year] = Π_{t=base+1..year} (1+infl[t])
    """
    years = sorted(annual_rates.keys())
    if base_year not in years:
        years.append(base_year)
        years = sorted(set(years))

    # 先构建一个“名义累乘”的 index（从最小年份起算），再归一化到 base_year=1
    start = min(years)
    idx = {start: 1.0}
    for y in range(start+1, max(years)+1):
        r = annual_rates.get(y, 0.0)
        idx[y] = idx[y-1] * (1.0 + r)

    # 归一化到 base_year
    base = idx[base_year]
    level = {y: idx[y]/base for y in idx}
    return level

def to_real(nominal_price: np.ndarray, years: np.ndarray, level: Dict[int, float], base_year: int) -> np.ndarray:
    """
    把名义价格换算到 base_year 的实际价格：
    Real = Nominal * (Level[base_year] / Level[year]) = Nominal / Level[year]  (因 Level[base]=1)
    """
    reals = []
    for p, y in zip(nominal_price, years):
        if y not in level:
            raise ValueError(f"年份 {y} 不在通胀表中，请补充 ANNUAL_INFL。")
        reals.append(float(p) / level[y])
    return np.array(reals, dtype=float)

def real_to_nominal(real_price: float, from_year: int, to_year: int, level: Dict[int, float]) -> float:
    """
    把 base_year 实际价换算成目标年份 to_year 的名义价。
    Real(base) -> Nominal(to) = Real * Level[to]
    （因为 Real = Nominal / Level[year]，当 Real 定义在 base_year 且 Level[base]=1）
    """
    return float(real_price) * level[to_year]

# ===== 5) 把历史名义价 → 实际价（统一到 BASE_REAL_YEAR 的购买力）=====
LEVEL = build_price_level_from_rates(ANNUAL_INFL, BASE_REAL_YEAR)

BOTTOMS_REAL = to_real(BOTTOMS_NOMINAL, BOTTOM_YEARS, LEVEL, BASE_REAL_YEAR)
TOPS_REAL    = to_real(TOPS_NOMINAL,    TOP_YEARS,    LEVEL, BASE_REAL_YEAR)
CURR_ANCHOR_REAL = to_real(np.array([CURR_PRICE_ANCHOR_NOMINAL]), np.array([CURR_ANCHOR_YEAR]), LEVEL, BASE_REAL_YEAR)[0]

# ===== 6) 四关系：比值历史（与通胀无关），用于“软约束" =====
mult_hist = np.array([TOPS_REAL[0]/BOTTOMS_REAL[0], TOPS_REAL[1]/BOTTOMS_REAL[1]])   # [~132, ~21] 近似不变
retr_hist = np.array([BOTTOMS_REAL[1]/TOPS_REAL[0], BOTTOMS_REAL[2]/TOPS_REAL[1]])   # [~0.16, ~0.22] 近似不变

# ===== 7) 在“Real 空间”做趋势拟合（只作软约束）=====
cycles_m = np.array([2.0, 3.0])  # 第二、第三轮
# 倍数衰减：log(mult) ~ a_m*cycle + b_m
a_m, b_m = np.polyfit(cycles_m, np.log(mult_hist), 1)
mult_pred_trend = float(np.exp(a_m*4 + b_m))   # 第四轮倍数趋势（soft）

# 回撤收敛：retr ~ a_r*cycle + b_r
a_r, b_r = np.polyfit(cycles_m, retr_hist, 1)
retr_pred_trend = float(a_r*4 + b_r)          # 第四轮回撤趋势（soft）

# 顶→顶（Real 空间）：log(top_real) ~ a_t*cycle + b_t
a_t, b_t = np.polyfit(cycles_m, np.log(TOPS_REAL), 1)
top_trend_4_real = float(np.exp(a_t*4 + b_t))

# 底→底（Real 空间）：log(bottom_real) ~ a_b*cycle + b_b
cycles_b = np.array([2.0, 3.0, 4.0])  # 2015, 2018, 2022
a_b, b_b = np.polyfit(cycles_b, np.log(BOTTOMS_REAL), 1)
bottom_trend_5_real = float(np.exp(a_b*5 + b_b))

# ===== 8) 在“Real 空间”做 M4 & R4 的联合搜索（四关系 + 现价/上轮顶硬约束）=====
B3_real = BOTTOMS_REAL[-1]        # 2022 实际底
prev_top_real = TOPS_REAL[-1]     # 2021 实际顶

# 约束权重（可调）
w_mult, w_retr, w_top, w_bottom, w_floor = 0.6, 1.0, 0.8, 0.08, 12.0
hard_floor = True

# 倍数搜索自适应下界：至少要高于“现价锚”（在 Real 空间下）
M_min = max(4.0, CURR_ANCHOR_REAL / B3_real, prev_top_real / B3_real)
mult_grid = np.linspace(M_min, 20.0, int((20.0 - M_min)/0.025) + 1)
retr_grid = np.linspace(0.26, 0.34, 161)  # 建议把回撤放在收敛区间附近

best = None
records = []

for M4 in mult_grid:
    T4_real = B3_real * M4
    # 硬约束：不得低于现价锚 & 上轮顶（都在 Real 空间比较）
    if hard_floor and (T4_real < CURR_ANCHOR_REAL or T4_real < prev_top_real):
        continue
    floor_pen = 0.0
    if (not hard_floor) and (T4_real < CURR_ANCHOR_REAL):
        floor_pen += w_floor * (np.log(CURR_ANCHOR_REAL) - np.log(T4_real))**2
    if (not hard_floor) and (T4_real < prev_top_real):
        floor_pen += w_floor * (np.log(prev_top_real) - np.log(T4_real))**2

    # 倍数与趋势（比值）
    cost_mult = w_mult * (np.log(M4) - np.log(mult_pred_trend))**2
    # 顶与顶趋势（Real 价格）
    cost_top  = w_top  * (np.log(T4_real) - np.log(top_trend_4_real))**2

    for R4 in retr_grid:
        B5_real = T4_real * R4
        # 回撤与趋势（比值）
        cost_retr = w_retr * (R4 - retr_pred_trend)**2
        # 底与底趋势（Real 价格）
        cost_bot  = w_bottom * (np.log(B5_real) - np.log(bottom_trend_5_real))**2

        cost = cost_mult + cost_retr + cost_top + cost_bot + floor_pen
        records.append((cost, M4, R4, T4_real, B5_real))
        if (best is None) or (cost < best[0]):
            best = (cost, M4, R4, T4_real, B5_real)

records = np.array(records, dtype=float)
if records.size == 0:
    raise RuntimeError("无可行解：检查通胀表、锚点、网格范围。")

min_cost = best[0]
band = records[records[:,0] <= (min_cost * 1.25)]  # “模糊正确”集合
T4_real_band = band[:,3]
B5_real_band = band[:,4]

# ===== 9) 最后一步：把 Real 预测转换为“名义价” =====
T4_nominal_best = real_to_nominal(best[3], BASE_REAL_YEAR, PRED_TOP_YEAR, LEVEL)
B5_nominal_best = real_to_nominal(best[4], BASE_REAL_YEAR, PRED_NEXT_BOTTOM_YEAR, LEVEL)

T4_nominal_band = np.array([real_to_nominal(x, BASE_REAL_YEAR, PRED_TOP_YEAR, LEVEL) for x in T4_real_band])
B5_nominal_band = np.array([real_to_nominal(x, BASE_REAL_YEAR, PRED_NEXT_BOTTOM_YEAR, LEVEL) for x in B5_real_band])

# 用分位数给出稳健区间
def band_str(vals: np.ndarray, qlo=0.10, qhi=0.90) -> Tuple[float, float]:
    return float(np.quantile(vals, qlo)), float(np.quantile(vals, qhi))

t4_lo, t4_hi = band_str(T4_nominal_band, 0.10, 0.90)
b5_lo, b5_hi = band_str(B5_nominal_band, 0.10, 0.90)

print("=== 四关系联合拟合（Real） → 历史全通胀校正 + 末端名义化 ===")
print(f"基准购买力年份: {BASE_REAL_YEAR}；预测顶年: {PRED_TOP_YEAR}；预测下一底年: {PRED_NEXT_BOTTOM_YEAR}")
print(f"最优(Real)：顶≈${best[3]:,.0f}，下一底≈${best[4]:,.0f}")
print(f"最优(Nominal)：顶≈${T4_nominal_best:,.0f}，下一底≈${B5_nominal_best:,.0f}")
print(f"模糊区间(名义顶,10~90分位)：${t4_lo:,.0f} ~ ${t4_hi:,.0f}")
print(f"模糊区间(名义底,10~90分位)：${b5_lo:,.0f} ~ ${b5_hi:,.0f}")


=== 四关系联合拟合（Real） → 历史全通胀校正 + 末端名义化 ===
基准购买力年份: 2025；预测顶年: 2025；预测下一底年: 2027
最优(Real)：顶≈$142,615，下一底≈$48,489
最优(Nominal)：顶≈$142,615，下一底≈$50,894
模糊区间(名义顶,10~90分位)：$128,961 ~ $204,061
模糊区间(名义底,10~90分位)：$40,109 ~ $66,083


In [52]:
# 你应该假设已知2025 的价格，然后抽出任何的价格，按照算法，反着算回去，如果也盐酸成功了， 就对了


In [53]:
# -*- coding: utf-8 -*-
from __future__ import annotations
from dataclasses import dataclass
from datetime import date, timedelta
from statistics import median
import numpy as np

# ===================== 1) 基础锚点（你之前给的） =====================
HALVINGS = [
    date(2012, 11, 28),
    date(2016, 7, 9),
    date(2020, 5, 11),
    date(2024, 4, 20),  # 最近一次
]
TOPS = [
    date(2013, 11, 29),
    date(2017, 12, 17),
    date(2021, 11, 10),
]
# 历史大顶的 AHR999（按月均值口径）
AHR999_PEAKS = {
    date(2013, 11, 27): 55.0,
    date(2017, 12, 19): 15.0,
    date(2021, 11, 8):  3.3,
}

# ===================== 2) 时间模型（模型 A） =====================
@dataclass
class TimeModelConfig:
    u_factor: float = 1.2       # 不确定度放大因子（覆盖UI波动）
    method: str = "regress"     # 'regress' 或 'median'

@dataclass
class TimeModelResult:
    center: date
    window_lo: date
    window_hi: date
    center_days: int
    corr_halving_vs_delay: float
    alt_center_by_median: date

def predict_peak_window_by_time(cfg: TimeModelConfig = TimeModelConfig()) -> TimeModelResult:
    # 历史“减半→大顶”的天数
    halving_to_top = np.array([(TOPS[i] - HALVINGS[i]).days for i in range(len(TOPS))], dtype=float)
    # 历史“减半→减半”的天数
    halving_intervals = np.array([(HALVINGS[i+1] - HALVINGS[i]).days for i in range(len(HALVINGS)-1)], dtype=float)

    corr = float(np.corrcoef(halving_intervals, halving_to_top)[0, 1]) if len(halving_intervals) >= 2 else 0.0

    if cfg.method == "regress" and len(halving_to_top) >= 2:
        # y = a*x + b
        x, y = halving_intervals, halving_to_top
        A = np.vstack([x, np.ones_like(x)]).T
        a, b = np.linalg.lstsq(A, y, rcond=None)[0]
        center_days = int(round(a * halving_intervals[-1] + b))
        resid = y - (a*x + b)
        sigma = float(np.sqrt(np.mean(resid**2))) if len(y) > 1 else float(np.std(y))
    else:
        center_days = int(round(median(halving_to_top)))
        sigma = float(np.std(halving_to_top))

    sigma *= cfg.u_factor
    anchor = HALVINGS[-1]
    center_dt = anchor + timedelta(days=center_days)
    window_lo = anchor + timedelta(days=int(round(center_days - sigma)))
    window_hi = anchor + timedelta(days=int(round(center_days + sigma)))
    alt_center = anchor + timedelta(days=int(round(median(halving_to_top))))

    return TimeModelResult(
        center=center_dt,
        window_lo=window_lo,
        window_hi=window_hi,
        center_days=center_days,
        corr_halving_vs_delay=corr,
        alt_center_by_median=alt_center
    )

# ===================== 3) AHR999 指数值预测（模型 C） =====================
@dataclass
class AHRModelResult:
    method: str
    k_or_slope: float
    intercept: float
    pred_2025: float
    rmse_log2: float

def _prepare_ahr_series():
    # 以 2013/2017/2021 作为第 1/2/3 个周期（避免和减半周期编号混淆）
    pts = sorted(AHR999_PEAKS.items(), key=lambda x: x[0])
    years = np.array([d.year for d,_ in pts], dtype=int)     # [2013, 2017, 2021]
    cycles = np.arange(1, len(years)+1, dtype=float)         # [1, 2, 3]
    vals = np.array([v for _,v in pts], dtype=float)         # [55, 15, 3.3]
    return cycles, vals

def fit_ahr_log2_linear() -> AHRModelResult:
    """
    纯数据拟合：对 log2(AHR) 做线性回归，外推到 cycle=4（即 2025）。
    """
    cycles, vals = _prepare_ahr_series()
    y = np.log2(vals)
    X = np.vstack([cycles, np.ones_like(cycles)]).T          # y = s * n + c
    s, c = np.linalg.lstsq(X, y, rcond=None)[0]
    pred_log2_2025 = s * 4.0 + c
    pred_2025 = float(2.0 ** pred_log2_2025)
    rmse = float(np.sqrt(np.mean((y - (s*cycles + c))**2)))
    return AHRModelResult(method="log2-linear", k_or_slope=s, intercept=c, pred_2025=pred_2025, rmse_log2=rmse)

def fit_ahr_base2_decay() -> AHRModelResult:
    """
    “以 2 为底”的递减模型（显式包含 2）：
    假设 log2(AHR) = a - k*(n-3)，使 2021 对齐，最小化对 2013/2017 的误差，解出 k。
    这是一个单参数（k）模型，a = log2(2021值)。
    """
    cycles, vals = _prepare_ahr_series()
    n1, n2, n3 = cycles        # 1, 2, 3
    v1, v2, v3 = vals          # 55, 15, 3.3

    a = np.log2(v3)            # 以 2021 为锚：n=3 时 log2(AHR)=a
    # 目标：min_k Σ (log2(v_hat_i) - log2(v_i))^2, i=1,2
    # 其中 v_hat_i = 2**(a + k*(3 - i))
    ks = np.linspace(0.5, 2.0, 3001)  # 搜索范围可调
    def err(k: float) -> float:
        v2_hat_log2 = a + k*(3 - 2)
        v1_hat_log2 = a + k*(3 - 1)*1.0
        e = (v2_hat_log2 - np.log2(v2))**2 + (v1_hat_log2 - np.log2(v1))**2
        return float(e)
    errs = np.array([err(k) for k in ks], dtype=float)
    k_best = float(ks[np.argmin(errs)])

    # 外推到 2025（n=4）：log2 = a - k*(4-3) = a - k
    pred_log2_2025 = a - k_best
    pred_2025 = float(2.0 ** pred_log2_2025)
    rmse = float(np.sqrt(np.min(errs)/2.0))  # 两个点的均方误差开方
    # 这里返回的 k 代表“每跨一轮衰减 2^k 倍”
    return AHRModelResult(method="base2-decay", k_or_slope=k_best, intercept=a, pred_2025=pred_2025, rmse_log2=rmse)

# ===================== 4) 运行与汇总 =====================
if __name__ == "__main__":
    # 时间预测
    t_cfg = TimeModelConfig(u_factor=1.2, method="regress")
    t_res = predict_peak_window_by_time(t_cfg)

    # AHR999 两种拟合
    m1 = fit_ahr_log2_linear()
    m2 = fit_ahr_base2_decay()

    # 选型逻辑：你可以按 RMSE 选，也可以两者都报给自己参考
    # 这里演示：优先选“纯数据回归”的结果作为主值，base2 作为校验
    primary = m1 if m1.rmse_log2 <= m2.rmse_log2 else m2
    secondary = m2 if primary is m1 else m1

    print("=== 时间模型（模型A） ===")
    print(f"减半→大顶相关系数 r: {t_res.corr_halving_vs_delay:.3f}")
    print(f"时间中心: {t_res.center}   （中位数备选中心: {t_res.alt_center_by_median}）")
    print(f"时间窗口: {t_res.window_lo} → {t_res.window_hi}")
    print()

    print("=== AHR999 指数值预测（模型C） ===")
    print(f"[主模型] {primary.method}: 预测 AHR999(2025) ≈ {primary.pred_2025:.2f} "
          f"(rmse_log2={primary.rmse_log2:.3f})")
    if primary.method == "log2-linear":
        print(f"    log2回归: slope={primary.k_or_slope:.3f}, intercept={primary.intercept:.3f}")
    else:
        print(f"    base2衰减: k={primary.k_or_slope:.3f} → 每轮衰减倍数≈2^{primary.k_or_slope:.3f}≈{2**primary.k_or_slope:.2f}x")
    print(f"[对照]   {secondary.method}: 预测 AHR999(2025) ≈ {secondary.pred_2025:.2f} "
          f"(rmse_log2={secondary.rmse_log2:.3f})")
    if secondary.method == "log2-linear":
        print(f"    log2回归: slope={secondary.k_or_slope:.3f}, intercept={secondary.intercept:.3f}")
    else:
        print(f"    base2衰减: k={secondary.k_or_slope:.3f} → 每轮衰减倍数≈2^{secondary.k_or_slope:.3f}≈{2**secondary.k_or_slope:.2f}x")


=== 时间模型（模型A） ===
减半→大顶相关系数 r: 0.980
时间中心: 2025-11-04   （中位数备选中心: 2025-09-28）
时间窗口: 2025-10-16 → 2025-11-23

=== AHR999 指数值预测（模型C） ===
[主模型] log2-linear: 预测 AHR999(2025) ≈ 0.84 (rmse_log2=0.073)
    log2回归: slope=-2.029, intercept=7.862
[对照]   base2-decay: 预测 AHR999(2025) ≈ 0.82 (rmse_log2=0.137)
    base2衰减: k=2.000 → 每轮衰减倍数≈2^2.000≈4.00x


In [56]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from scipy.stats import norm

# === 输入：四组模型结果 ===
time_points = [
    ("Baseline", "2025-09-28", 1.0),
    ("Regression", "2025-11-04", 1.5),  # 权重可以根据rmse调整
    ("Top2Top", "2025-11-01", 1.2),
    ("Combined", "2025-10-16", 1.0),    # 中间参考点
]

price_ranges = {
    "ModelA": {"peak": (122062, 122062), "bottom": (42722, 42722)},
    "ModelB": {"peak": (133381, 133381), "bottom": (48369, 48369)},
    "ModelC": {"peak": (142615, 142615), "bottom": (50894, 50894)},
    "ModelD": {"peak": (128961, 204061), "bottom": (40109, 66083)},  # 区间型
}

# === 时间综合 ===
def combine_time(points):
    dates = []
    weights = []
    for name, date_str, w in points:
        dt = datetime.strptime(date_str, "%Y-%m-%d")
        dates.append(dt.toordinal())  # 转换为整数天
        weights.append(w)
    dates = np.array(dates)
    weights = np.array(weights)
    avg = np.average(dates, weights=weights)
    std = np.sqrt(np.average((dates-avg)**2, weights=weights))
    avg_date = datetime.fromordinal(int(avg))
    lower = datetime.fromordinal(int(avg-std))
    upper = datetime.fromordinal(int(avg+std))
    return avg_date, lower, upper

# === 价格综合 ===
def combine_price(price_dict):
    peaks = []
    bottoms = []
    for m, vals in price_dict.items():
        peak_low, peak_high = vals["peak"]
        bottom_low, bottom_high = vals["bottom"]
        peaks.append((peak_low+peak_high)/2)
        bottoms.append((bottom_low+bottom_high)/2)
    return (np.mean(peaks), np.min(peaks), np.max(peaks)), \
           (np.mean(bottoms), np.min(bottoms), np.max(bottoms))

# === 执行 ===
avg_date, t_low, t_high = combine_time(time_points)
(peak_avg, peak_min, peak_max), (bottom_avg, bottom_min, bottom_max) = combine_price(price_ranges)

print("=== 综合预测结果 ===")
print(f"时间窗口: {t_low.date()} → {t_high.date()} (均值中心: {avg_date.date()})")
print(f"价格区间(顶): 平均={peak_avg:,.0f}, 范围={peak_min:,.0f} ~ {peak_max:,.0f}")
print(f"价格区间(底): 平均={bottom_avg:,.0f}, 范围={bottom_min:,.0f} ~ {bottom_max:,.0f}")


=== 综合预测结果 ===
时间窗口: 2025-10-07 → 2025-11-05 (均值中心: 2025-10-22)
价格区间(顶): 平均=141,142, 范围=122,062 ~ 166,511
价格区间(底): 平均=48,770, 范围=42,722 ~ 53,096


In [57]:
# -*- coding: utf-8 -*-
from __future__ import annotations
from datetime import date, datetime, timedelta
from dataclasses import dataclass
import numpy as np

# =========================
# 1) 你提供的观测（时间 & 价格）
# =========================

# 时间观测（名称, 日期字符串, 权重, 观测类型）
# 这里我们用三个中心点：中位数、回归、顶→顶外推
time_points = [
    ("MedianCenter",    "2025-09-28", 1.0, "point"),
    ("RegressionCenter","2025-11-04", 1.3, "point"),
    ("TopToTopCenter",  "2025-11-01", 1.1, "point"),
]

# 时间硬边界窗口（来自你之前的时间模型 A）
time_hard_window = ("2025-10-16", "2025-11-23")

# 价格观测（名称=>顶/底区间或点估）
# 其中 D 模型是区间（10~90分位），其余是点估
price_obs = {
    "ModelA_nominal_1%":   {"peak": (122_062, 122_062)},                 # 点估
    "ModelB_nominal_3%":   {"peak": (133_381, 133_381)},                 # 点估
    "ModelC_real_base2025":{"peak": (142_615, 142_615)},                 # 点估
    "ModelD_band_nominal": {"peak": (128_961, 204_061), "is_band": True} # 区间(10~90分位)
}
# 底部区间仅作回显和 sanity check，不参与优化（也可以参与，思路相同）
bottom_band_nominal = (40_109, 66_083)

# AHR999 参考（来自你最新拟合）
ahr999_pred = {
    "log2_linear": 0.84,
    "base2_decay": 0.82,
    "note": "AHR999不直接约束价格，仅做旁证。"
}

# =========================
# 2) 小工具
# =========================

def to_ordinal(s: str) -> int:
    return datetime.strptime(s, "%Y-%m-%d").toordinal()

def from_ordinal(o: int) -> date:
    return datetime.fromordinal(o).date()

def sigma_from_band_10_90(lo: float, hi: float) -> float:
    """
    若传入的是10~90分位区间，则可近似换算为正态的σ：
    p10 = -1.28155σ, p90 = +1.28155σ -> 宽度 = 2*1.28155σ
    """
    return (hi - lo) / (2.0 * 1.28155)

# =========================
# 3) 构造观测的均值与σ
# =========================

# 时间观测：把三个中心点转成天数（以ordinal表示），并给每个点一个σ_time
# σ_time 可按经验设定（回归较稳，给更小σ；中位数稍大）
time_mu = []
time_sigma = []
time_w = []

for name, dstr, w, kind in time_points:
    mu = to_ordinal(dstr)
    if name == "RegressionCenter":
        sigma = 18.0  # 回归中心更稳（可调）
    elif name == "TopToTopCenter":
        sigma = 20.0
    else:
        sigma = 26.0  # 中位数略宽
    time_mu.append(mu)
    time_sigma.append(sigma)
    time_w.append(w)

time_mu = np.array(time_mu, dtype=float)
time_sigma = np.array(time_sigma, dtype=float)
time_w = np.array(time_w, dtype=float)

# 时间硬边界
T_low_hard = to_ordinal(time_hard_window[0])
T_hi_hard  = to_ordinal(time_hard_window[1])

# 价格观测：将“点估”转为(μ, σ)；将“区间”转为(μ≈中位, σ≈由10~90分位换算)
peak_mus = []
peak_sigmas = []
peak_ws = []

default_rel_sigma = 0.08  # 点估的默认相对不确定度（8%，可调）
for name, d in price_obs.items():
    lo, hi = d["peak"]
    if d.get("is_band", False):
        mu = 0.5 * (lo + hi)   # 取区间中点作为μ
        sigma = sigma_from_band_10_90(lo, hi)
        w = 1.0
    else:
        mu = lo  # lo==hi即点估
        sigma = max(500.0, default_rel_sigma * mu)  # 至少给个固定下限，避免σ过小
        # 越新越可信，可适当加权；这里给 Real 基准的点更高权重
        if "real_base2025" in name:
            w = 1.3
        elif "nominal_3%" in name:
            w = 1.1
        else:
            w = 1.0

    peak_mus.append(mu)
    peak_sigmas.append(sigma)
    peak_ws.append(w)

peak_mus = np.array(peak_mus, dtype=float)
peak_sigmas = np.array(peak_sigmas, dtype=float)
peak_ws = np.array(peak_ws, dtype=float)

# 价格搜索范围（自动合并各观测的最小/最大）
P_search_lo = float(min([v["peak"][0] for v in price_obs.values()]))
P_search_hi = float(max([v["peak"][1] for v in price_obs.values()]))

# 为了给一点外延，放宽5%
pad = 0.05
P_search_lo *= (1.0 - pad)
P_search_hi *= (1.0 + pad)

# =========================
# 4) 目标函数 & 网格搜索
# =========================

def cost_time(T: float) -> float:
    # Σ w * ((T - μ)/σ)^2
    z2 = ((T - time_mu) / time_sigma) ** 2
    return float(np.sum(time_w * z2))

def cost_peak(P: float) -> float:
    z2 = ((P - peak_mus) / peak_sigmas) ** 2
    return float(np.sum(peak_ws * z2))

def total_cost(T: float, P: float) -> float:
    return cost_time(T) + cost_peak(P)

# 网格（时间按天，价格按步长 500 美元；都可调）
T_grid = np.arange(T_low_hard, T_hi_hard + 1, 1, dtype=float)
P_grid = np.arange(P_search_lo, P_search_hi + 1, 500.0, dtype=float)

best = None
band_records = []

for T in T_grid:
    ct = cost_time(T)
    # 先粗筛时间成本，避免不必要的价格循环（可选小优化）
    for P in P_grid:
        c = ct + cost_peak(P)
        band_records.append((c, T, P))
        if (best is None) or (c < best[0]):
            best = (c, T, P)

band = np.array(band_records, dtype=float)
minc = best[0]

# 取“接近最优”的一团解（阈值可调：相对阈 or 绝对阈）
rel_eps = 0.20  # 允许总成本比最优高 20%
mask = band[:,0] <= (minc * (1.0 + rel_eps))
band_sel = band[mask]

# 融合时间区间与价格区间（用所选“模糊解”的 min/max）
T_lo = int(np.min(band_sel[:,1]));  T_hi = int(np.max(band_sel[:,1]))
P_lo = float(np.min(band_sel[:,2])); P_hi = float(np.max(band_sel[:,2]))

# 中心取“权重均值”（在模糊集合内加权）
def weighted_center(vals: np.ndarray, costs: np.ndarray) -> float:
    # 代价越小权重越大：w = 1 / (1 + cost - minc)
    w = 1.0 / (1.0 + (costs - minc))
    return float(np.sum(vals * w) / np.sum(w))

T_center = int(round(weighted_center(band_sel[:,1], band_sel[:,0])))
P_center = float(weighted_center(band_sel[:,2], band_sel[:,0]))

# =========================
# 5) 输出
# =========================
print("=== 强制联合拟合（Multi-Model Fusion） ===")
print(f"时间硬边界      : {from_ordinal(T_low_hard)} → {from_ordinal(T_hi_hard)}")
print(f"融合时间中心    : {from_ordinal(T_center)}")
print(f"融合时间区间    : {from_ordinal(T_lo)} → {from_ordinal(T_hi)}")
print()
print(f"价格搜索范围    : ${P_search_lo:,.0f} → ${P_search_hi:,.0f}")
print(f"融合价格中心    : ${P_center:,.0f}")
print(f"融合价格区间    : ${P_lo:,.0f} → ${P_hi:,.0f}")
print()
print("— 观测回显（时间点, σ, 权重）—")
for (name, dstr, w, _), sig in zip(time_points, time_sigma):
    print(f"{name:<16} {dstr}   σ≈{sig:.1f}天   w={w:.2f}")
print()
print("— 观测回显（价格）—")
for name, d in price_obs.items():
    lo, hi = d["peak"]
    if d.get("is_band", False):
        sig = sigma_from_band_10_90(lo, hi)
        print(f"{name:<20} peak区间: ${lo:,.0f}~${hi:,.0f}  σ≈{sig:,.0f}")
    else:
        sig = max(500.0, default_rel_sigma * lo)
        print(f"{name:<20} peak点估: ${lo:,.0f}         σ≈{sig:,.0f}")
print()
print("— AHR999 旁证 —")
print(f"log2-linear≈{ahr999_pred['log2_linear']:.2f}, base2-decay≈{ahr999_pred['base2_decay']:.2f}  （不直接入目标，仅旁证）")


=== 强制联合拟合（Multi-Model Fusion） ===
时间硬边界      : 2025-10-16 → 2025-11-23
融合时间中心    : 2025-10-27
融合时间区间    : 2025-10-17 → 2025-11-07

价格搜索范围    : $115,959 → $214,264
融合价格中心    : $133,754
融合价格区间    : $128,459 → $138,959

— 观测回显（时间点, σ, 权重）—
MedianCenter     2025-09-28   σ≈26.0天   w=1.00
RegressionCenter 2025-11-04   σ≈18.0天   w=1.30
TopToTopCenter   2025-11-01   σ≈20.0天   w=1.10

— 观测回显（价格）—
ModelA_nominal_1%    peak点估: $122,062         σ≈9,765
ModelB_nominal_3%    peak点估: $133,381         σ≈10,670
ModelC_real_base2025 peak点估: $142,615         σ≈11,409
ModelD_band_nominal  peak区间: $128,961~$204,061  σ≈29,300

— AHR999 旁证 —
log2-linear≈0.84, base2-decay≈0.82  （不直接入目标，仅旁证）
